<a href="https://colab.research.google.com/github/Noors-lab/VigIQ-Vigilance-Intelligence-/blob/main/Dashboard.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
!pip install fastapi uvicorn python-multipart aiofiles pyngrok -q
!ngrok config add-authtoken 3FOsYA6oWkyqkHmDa0NasWeOyzy_FkQDEprTQ7yrshVbGwGC
print("Done.")

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml
Done.


In [11]:
import os
import json
import asyncio
import threading
import time
from pathlib import Path
from fastapi import FastAPI
from fastapi.responses import HTMLResponse, FileResponse
from fastapi.staticfiles import StaticFiles
import uvicorn
from pyngrok import ngrok

# ─── Setup ───
ALERTS_DIR = "/content/vigiq_alerts"
os.makedirs(ALERTS_DIR, exist_ok=True)

app = FastAPI()

# ─── API Routes ───
@app.get("/api/alerts")
def get_alerts():
    log_path = os.path.join(ALERTS_DIR, "alert_log.json")
    if not os.path.exists(log_path):
        return []
    with open(log_path) as f:
        return json.load(f)

@app.post("/api/alerts/{alert_id}/confirm")
def confirm_alert(alert_id: int):
    return update_alert_status(alert_id, "confirmed")

@app.post("/api/alerts/{alert_id}/dismiss")
def dismiss_alert(alert_id: int):
    return update_alert_status(alert_id, "dismissed")

def update_alert_status(alert_id, status):
    log_path = os.path.join(ALERTS_DIR, "alert_log.json")
    with open(log_path) as f:
        alerts = json.load(f)
    for alert in alerts:
        if alert.get("person_id") == alert_id:
            alert["status"] = status
    with open(log_path, "w") as f:
        json.dump(alerts, f, indent=2)
    return {"success": True, "status": status}

@app.get("/api/clip/{clip_name}")
def get_clip(clip_name: str):
    clip_path = os.path.join(ALERTS_DIR, clip_name)
    if os.path.exists(clip_path):
        return FileResponse(clip_path, media_type="video/mp4")
    return {"error": "clip not found"}

# ─── Dashboard HTML ───
@app.get("/", response_class=HTMLResponse)
def dashboard():
    return """
<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>VigIQ — Loss Prevention Dashboard</title>
<style>
  * { margin: 0; padding: 0; box-sizing: border-box; }

  body {
    font-family: 'Segoe UI', Arial, sans-serif;
    background: #FFFFFF;
    color: #1A1A1A;
    min-height: 100vh;
  }

  /* ─── Header ─── */
  header {
    background: #FFFFFF;
    border-bottom: 3px solid #CC0000;
    padding: 16px 32px;
    display: flex;
    align-items: center;
    justify-content: space-between;
    position: sticky;
    top: 0;
    z-index: 100;
    box-shadow: 0 2px 8px rgba(0,0,0,0.08);
  }

  .logo {
    font-size: 24px;
    font-weight: 800;
    color: #CC0000;
    letter-spacing: 2px;
  }

  .logo span {
    color: #1A1A1A;
  }

  .status-bar {
    display: flex;
    align-items: center;
    gap: 24px;
  }

  .live-dot {
    width: 10px;
    height: 10px;
    background: #00C851;
    border-radius: 50%;
    display: inline-block;
    margin-right: 6px;
    animation: pulse 1.5s infinite;
  }

  @keyframes pulse {
    0%   { opacity: 1; }
    50%  { opacity: 0.3; }
    100% { opacity: 1; }
  }

  .live-text {
    font-size: 13px;
    color: #555;
    font-weight: 600;
  }

  /* ─── Stats bar ─── */
  .stats-bar {
    background: #F8F8F8;
    border-bottom: 1px solid #E0E0E0;
    padding: 12px 32px;
    display: flex;
    gap: 40px;
  }

  .stat {
    display: flex;
    flex-direction: column;
  }

  .stat-label {
    font-size: 11px;
    color: #888;
    text-transform: uppercase;
    letter-spacing: 1px;
  }

  .stat-value {
    font-size: 22px;
    font-weight: 700;
    color: #1A1A1A;
  }

  .stat-value.red   { color: #CC0000; }
  .stat-value.green { color: #00A841; }
  .stat-value.grey  { color: #888; }

  /* ─── Main layout ─── */
  .main {
    display: grid;
    grid-template-columns: 1fr 1fr;
    gap: 24px;
    padding: 24px 32px;
    max-width: 1400px;
    margin: 0 auto;
  }

  /* ─── Camera section ─── */
  .section-title {
    font-size: 13px;
    font-weight: 700;
    text-transform: uppercase;
    letter-spacing: 1.5px;
    color: #888;
    margin-bottom: 12px;
  }

  .camera-grid {
    display: grid;
    grid-template-columns: 1fr 1fr;
    gap: 12px;
  }

  .camera-card {
    background: #1A1A1A;
    border-radius: 8px;
    overflow: hidden;
    position: relative;
    aspect-ratio: 16/9;
  }

  .camera-card .cam-label {
    position: absolute;
    bottom: 8px;
    left: 8px;
    background: rgba(0,0,0,0.7);
    color: white;
    font-size: 11px;
    padding: 3px 8px;
    border-radius: 4px;
    font-weight: 600;
  }

  .camera-placeholder {
    width: 100%;
    height: 100%;
    display: flex;
    align-items: center;
    justify-content: center;
    flex-direction: column;
    gap: 8px;
    color: #555;
    font-size: 12px;
  }

  .camera-placeholder .cam-icon {
    font-size: 28px;
  }

  /* ─── Alert section ─── */
  .alerts-panel {
    display: flex;
    flex-direction: column;
    gap: 12px;
    max-height: 75vh;
    overflow-y: auto;
  }

  .alerts-panel::-webkit-scrollbar { width: 4px; }
  .alerts-panel::-webkit-scrollbar-thumb { background: #CC0000; border-radius: 4px; }

  .alert-card {
    background: #FFFFFF;
    border: 1px solid #E8E8E8;
    border-left: 4px solid #CC0000;
    border-radius: 8px;
    padding: 16px;
    transition: box-shadow 0.2s;
  }

  .alert-card:hover {
    box-shadow: 0 4px 16px rgba(204,0,0,0.1);
  }

  .alert-card.confirmed {
    border-left-color: #00A841;
    opacity: 0.7;
  }

  .alert-card.dismissed {
    border-left-color: #CCCCCC;
    opacity: 0.5;
  }

  .alert-header {
    display: flex;
    justify-content: space-between;
    align-items: flex-start;
    margin-bottom: 10px;
  }

  .alert-title {
    font-size: 13px;
    font-weight: 700;
    color: #CC0000;
    text-transform: uppercase;
    letter-spacing: 0.5px;
  }

  .alert-time {
    font-size: 11px;
    color: #888;
  }

  .alert-reason {
    font-size: 13px;
    color: #333;
    margin-bottom: 8px;
  }

  .alert-meta {
    display: flex;
    gap: 16px;
    margin-bottom: 12px;
  }

  .meta-item {
    font-size: 11px;
    color: #888;
  }

  .meta-item span {
    color: #1A1A1A;
    font-weight: 600;
  }

  /* ─── Video clip ─── */
  .clip-container {
    background: #1A1A1A;
    border-radius: 6px;
    overflow: hidden;
    margin-bottom: 12px;
    cursor: pointer;
    position: relative;
  }

  .clip-container video {
    width: 100%;
    display: block;
  }

  .play-overlay {
    position: absolute;
    top: 50%;
    left: 50%;
    transform: translate(-50%, -50%);
    background: rgba(204,0,0,0.85);
    color: white;
    width: 40px;
    height: 40px;
    border-radius: 50%;
    display: flex;
    align-items: center;
    justify-content: center;
    font-size: 16px;
    pointer-events: none;
    transition: opacity 0.2s;
  }

  .clip-container:hover .play-overlay { opacity: 0; }

  /* ─── Buttons ─── */
  .alert-actions {
    display: flex;
    gap: 8px;
  }

  .btn {
    flex: 1;
    padding: 8px;
    border: none;
    border-radius: 6px;
    font-size: 12px;
    font-weight: 700;
    cursor: pointer;
    letter-spacing: 0.5px;
    transition: all 0.2s;
  }

  .btn-confirm {
    background: #CC0000;
    color: white;
  }

  .btn-confirm:hover { background: #AA0000; }

  .btn-dismiss {
    background: #F0F0F0;
    color: #555;
  }

  .btn-dismiss:hover { background: #E0E0E0; }

  /* ─── Empty state ─── */
  .empty-state {
    text-align: center;
    padding: 60px 20px;
    color: #AAAAAA;
  }

  .empty-state .icon { font-size: 48px; margin-bottom: 12px; }
  .empty-state p { font-size: 14px; }

  /* ─── Status badge ─── */
  .status-badge {
    font-size: 10px;
    font-weight: 700;
    padding: 3px 8px;
    border-radius: 12px;
    text-transform: uppercase;
    letter-spacing: 0.5px;
  }

  .badge-confirmed { background: #E8F5E9; color: #00A841; }
  .badge-dismissed { background: #F5F5F5; color: #888; }
  .badge-pending   { background: #FFEBEE; color: #CC0000; }
</style>
</head>
<body>

<!-- Header -->
<header>
  <div class="logo">VIG<span>IQ</span></div>
  <div class="status-bar">
    <div class="live-text">
      <span class="live-dot"></span>SYSTEM ACTIVE
    </div>
    <div class="live-text" id="clock"></div>
  </div>
</header>

<!-- Stats bar -->
<div class="stats-bar">
  <div class="stat">
    <div class="stat-label">Total Alerts</div>
    <div class="stat-value red" id="total-alerts">0</div>
  </div>
  <div class="stat">
    <div class="stat-label">Confirmed</div>
    <div class="stat-value green" id="confirmed-count">0</div>
  </div>
  <div class="stat">
    <div class="stat-label">Dismissed</div>
    <div class="stat-value grey" id="dismissed-count">0</div>
  </div>
  <div class="stat">
    <div class="stat-label">Pending Review</div>
    <div class="stat-value" id="pending-count">0</div>
  </div>
</div>

<!-- Main -->
<div class="main">

  <!-- Left: Cameras -->
  <div>
    <div class="section-title">📷 Live Cameras</div>
    <div class="camera-grid" id="camera-grid">
      <div class="camera-card">
        <div class="camera-placeholder">
          <div class="cam-icon">📷</div>
          <div>Camera 01</div>
        </div>
        <div class="cam-label">CAM 01 — Entrance</div>
      </div>
      <div class="camera-card">
        <div class="camera-placeholder">
          <div class="cam-icon">📷</div>
          <div>Camera 02</div>
        </div>
        <div class="cam-label">CAM 02 — Aisle A</div>
      </div>
      <div class="camera-card">
        <div class="camera-placeholder">
          <div class="cam-icon">📷</div>
          <div>Camera 03</div>
        </div>
        <div class="cam-label">CAM 03 — Checkout</div>
      </div>
      <div class="camera-card">
        <div class="camera-placeholder">
          <div class="cam-icon">📷</div>
          <div>Camera 04</div>
        </div>
        <div class="cam-label">CAM 04 — Aisle B</div>
      </div>
    </div>
  </div>

  <!-- Right: Alerts -->
  <div>
    <div class="section-title">🚨 Alert Feed</div>
    <div class="alerts-panel" id="alerts-panel">
      <div class="empty-state">
        <div class="icon">✓</div>
        <p>No alerts yet. System is monitoring.</p>
      </div>
    </div>
  </div>

</div>

<script>
  // ─── Clock ───
  function updateClock() {
    const now = new Date();
    document.getElementById('clock').textContent =
      now.toLocaleTimeString('en-US', {hour12: false});
  }
  updateClock();
  setInterval(updateClock, 1000);

  // ─── Load alerts ───
  let knownAlerts = new Set();

  async function loadAlerts() {
    try {
      const res  = await fetch('/api/alerts');
      const data = await res.json();

      let total     = data.length;
      let confirmed = data.filter(a => a.status === 'confirmed').length;
      let dismissed = data.filter(a => a.status === 'dismissed').length;
      let pending   = total - confirmed - dismissed;

      document.getElementById('total-alerts').textContent    = total;
      document.getElementById('confirmed-count').textContent = confirmed;
      document.getElementById('dismissed-count').textContent = dismissed;
      document.getElementById('pending-count').textContent   = pending;

      if (total === 0) return;

      const panel = document.getElementById('alerts-panel');
      panel.innerHTML = '';

      // Latest first
      const sorted = [...data].reverse();

      sorted.forEach((alert, idx) => {
        const status  = alert.status || 'pending';
        const clipUrl = alert.clip ? '/api/clip/' + alert.clip.split('/').pop() : null;

        const card = document.createElement('div');
        card.className = `alert-card ${status}`;
        card.id = `alert-${alert.person_id}`;

        card.innerHTML = `
          <div class="alert-header">
            <div class="alert-title">⚠ Person ${alert.person_id}</div>
            <div style="display:flex;gap:8px;align-items:center;">
              <span class="status-badge badge-${status}">
                ${status === 'pending' ? 'Review' : status}
              </span>
              <div class="alert-time">${alert.time || ''}</div>
            </div>
          </div>
          <div class="alert-reason">${alert.reason}</div>
          <div class="alert-meta">
            <div class="meta-item">Confidence: <span>${(alert.confidence*100).toFixed(0)}%</span></div>
            <div class="meta-item">Timestamp: <span>${alert.timestamp}s</span></div>
          </div>
          ${clipUrl ? `
          <div class="clip-container" onclick="playClip(this)">
            <video preload="metadata">
              <source src="${clipUrl}" type="video/mp4">
            </video>
            <div class="play-overlay">▶</div>
          </div>` : ''}
          <div class="alert-actions">
            <button class="btn btn-confirm"
              onclick="updateStatus(${alert.person_id}, 'confirm', this)">
              ✓ CONFIRM THEFT
            </button>
            <button class="btn btn-dismiss"
              onclick="updateStatus(${alert.person_id}, 'dismiss', this)">
              ✕ DISMISS
            </button>
          </div>
        `;
        panel.appendChild(card);
      });

    } catch(e) {
      console.error('Failed to load alerts:', e);
    }
  }

  // ─── Play clip on click ───
  function playClip(container) {
    const video = container.querySelector('video');
    if (video.paused) {
      video.play();
    } else {
      video.pause();
    }
  }

  // ─── Confirm / Dismiss ───
  async function updateStatus(personId, action, btn) {
    const res = await fetch(`/api/alerts/${personId}/${action}`, {method:'POST'});
    const data = await res.json();
    if (data.success) {
      loadAlerts();
    }
  }

  // ─── Auto refresh every 5 seconds ───
  loadAlerts();
  setInterval(loadAlerts, 5000);
</script>
</body>
</html>
"""

# ─── Run server with ngrok ───
def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="error")

thread = threading.Thread(target=run_server, daemon=True)
thread.start()
time.sleep(2)

# Open ngrok tunnel
public_url = ngrok.connect(8000)
print(f"\n{'='*50}")
print(f"VigIQ Dashboard is LIVE")
print(f"Open this URL: {public_url}")
print(f"{'='*50}\n")

ERROR:    [Errno 98] error while attempting to bind on address ('0.0.0.0', 8000): address already in use



VigIQ Dashboard is LIVE
Open this URL: NgrokTunnel: "https://kindness-wake-frequency.ngrok-free.dev" -> "http://localhost:8000"



In [12]:
import shutil
import os
from google.colab import drive

drive.mount('/content/drive')

# Copy alerts from Drive to local alerts dir
drive_alerts = "/content/drive/MyDrive/shoplifting/vigiq_alerts"
local_alerts = "/content/vigiq_alerts"

os.makedirs(local_alerts, exist_ok=True)

for f in os.listdir(drive_alerts):
    src = os.path.join(drive_alerts, f)
    dst = os.path.join(local_alerts, f)
    shutil.copy2(src, dst)
    print(f"Copied: {f}")

print("\nDone. Refresh the dashboard now.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Copied: alert_log.json
Copied: alert_1_7s.mp4

Done. Refresh the dashboard now.


In [13]:
import json
import os

alerts_dir = "/content/vigiq_alerts"
log_path = os.path.join(alerts_dir, "alert_log.json")

with open(log_path) as f:
    alerts = json.load(f)

# Fix clip paths — just keep filename not full path
for alert in alerts:
    if "clip" in alert:
        alert["clip"] = os.path.basename(alert["clip"])

with open(log_path, "w") as f:
    json.dump(alerts, f, indent=2)

print("Fixed. Alert log updated:")
print(json.dumps(alerts, indent=2))

Fixed. Alert log updated:
[
  {
    "person_id": 1,
    "reason": "suspicious motion detected",
    "confidence": 0.88,
    "timestamp": 7.0,
    "clip": "alert_1_7s.mp4",
    "time": "2026-06-27 13:48:37"
  }
]


In [14]:
import json
import os

alerts_dir = "/content/vigiq_alerts"
log_path = os.path.join(alerts_dir, "alert_log.json")

# Check alert log
with open(log_path) as f:
    alerts = json.load(f)

print("Alert log:")
print(json.dumps(alerts, indent=2))

print("\nFiles in alerts dir:")
for f in os.listdir(alerts_dir):
    size = os.path.getsize(os.path.join(alerts_dir, f))
    print(f"  {f} ({size/1024:.1f} KB)")

Alert log:
[
  {
    "person_id": 1,
    "reason": "suspicious motion detected",
    "confidence": 0.88,
    "timestamp": 7.0,
    "clip": "alert_1_7s.mp4",
    "time": "2026-06-27 13:48:37"
  }
]

Files in alerts dir:
  alert_log.json (0.2 KB)
  alert_1_7s.mp4 (1660.3 KB)


In [15]:
import subprocess
import os

input_clip = "/content/vigiq_alerts/alert_1_7s.mp4"
output_clip = "/content/vigiq_alerts/alert_1_7s_web.mp4"

# Re-encode to H264 for browser compatibility
subprocess.run([
    'ffmpeg', '-i', input_clip,
    '-vcodec', 'libx264',
    '-acodec', 'aac',
    '-y', output_clip
], capture_output=True)

print(f"Done. Size: {os.path.getsize(output_clip)/1024:.1f} KB")

# Update alert log to use new clip
import json
log_path = "/content/vigiq_alerts/alert_log.json"
with open(log_path) as f:
    alerts = json.load(f)

alerts[0]["clip"] = "alert_1_7s_web.mp4"

with open(log_path, "w") as f:
    json.dump(alerts, f, indent=2)

print("Alert log updated.")

Done. Size: 554.9 KB
Alert log updated.


In [16]:
import os
from google.colab import drive

drive.mount('/content/drive')

# Save dashboard code
dashboard_dir = "/content/drive/MyDrive/shoplifting/vigiq_app"
os.makedirs(dashboard_dir, exist_ok=True)

# Save the full pipeline + dashboard as one file
code = '''
# VigIQ — Full Pipeline + Dashboard
# Run this file to start VigIQ

# Install dependencies first:
# pip install fastapi uvicorn python-multipart aiofiles pyngrok ultralytics torch opencv-python

# Then run:
# python vigiq_app.py
'''

with open(os.path.join(dashboard_dir, "README.txt"), "w") as f:
    f.write(code)

print("Saved to Drive.")
print(f"Location: {dashboard_dir}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Saved to Drive.
Location: /content/drive/MyDrive/shoplifting/vigiq_app


In [17]:
import os
from google.colab import drive

drive.mount('/content/drive')

base = "/content/drive/MyDrive/shoplifting/vigiq_github"
dirs = [
    f"{base}/src",
    f"{base}/models",
    f"{base}/notebooks",
]

for d in dirs:
    os.makedirs(d, exist_ok=True)

print("Structure created.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Structure created.
